# Interactive Notebook 01 - Introduction to PWM:

This interactive jupyter notebook is intended as an introduction to the basics of Pulse width modulation (and to the basics of python programming and jupyter notebooks). 

For help with the installation of the required software, consider the comments in ```CTPD_course\interactive_notebooks\README.md```.
Throughout the exercises, we will be using a combination of scientific computation libraries from the [JAX](https://docs.jax.dev/en/latest/notebooks/thinking_in_jax.html) ecosystem and visualize them with [matplotlib](https://matplotlib.org/) and [ipywidgets](https://ipywidgets.readthedocs.io/en/stable/).

### Preliminaries & Imports:

In [ ]:
# automatically reloads imported ```.py```-files once they are changed and saved
%load_ext autoreload
%autoreload 2

In [ ]:
# imports required packages
from functools import partial
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import rc
mpl.rcParams.update({'font.size': 20})
import jax
import jax.numpy as jnp

(**Optional**: If you have LaTeX installed, you can use the following lines for pretty rendering of plot labels.
Any LaTeX installation should work, as long as all the required packages are installed, e.g., [MiKTeX](https://miktex.org/) or [TeXLive](https://www.tug.org/texlive/))

In [ ]:
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

---
## Signal generation example:

To get a first feeling for signal processing in Python, let us build the PWM carrier signal step-by-step and visualize it.

The following creates a discretized time array with $t \in [0, 20T_\mathrm{p})$ consisting of $10000$ elements. In this context, $f_{c} = 50 \mathrm{Hz}$ is the carrier frequency and $T_\mathrm{p} = 1 / f_{c}$ the pulse period.

In [ ]:
carrier_freq = 10  # Hz
T_p = 1 / carrier_freq
t = jnp.linspace(0, 10*T_p, 10000)
print(t.shape)

This creates an array with $10000$ elements, that is a discretization of a sinusoidal signal with $f_{c}$ and amplitude $1$:

In [ ]:
x_t = jnp.sin(2 * jnp.pi * carrier_freq * t)
print(x_t.shape)

In [ ]:
# Visualization:
fig, ax = plt.subplots(1,1, figsize=(8,4))

ax.plot(t, x_t)

ax.grid(alpha=0.5)
ax.set_ylabel("$x(t)$")
ax.set_xlabel("$t$ in s")
ax.set_ylim(-1.1, 1.1)
ax.set_xlim(t[0], t[-1])
ax.tick_params(which="major", axis="y", direction="in")
ax.tick_params(which="both", axis="x", direction="in")

plt.show()

Now we are implementing a function for the generation of triangular signals.
The steps are:
- Normalization of time with the pulse period, i.e., $\tilde{t} = t / T_\mathrm{p}$
- Phase shift of normalized time with $\varphi / (2\pi)$
- Modulo operation due to periodicity with normalized period $\tilde{T} = 1.0$
- For the first half of each period, the signal is $1.0 - 4.0 \cdot \tilde{t}$
- for the second half, it is $4.0 \cdot \tilde{t} - 3.0$

In [ ]:
def triangular_signal(t, frequency=1.0, amplitude=1.0, phase=0.0):

    T_p = 1.0 / frequency
    t_normalized = t / T_p
    t_normalized = t_normalized + phase / (2 * jnp.pi)
    t_normalized = t_normalized % 1.0

    triangle = jnp.where(
        t_normalized < 0.5,
        1.0 - 4.0 * t_normalized,
        4.0 * t_normalized - 3.0,
    )

    return amplitude * triangle

In [ ]:
c_t = triangular_signal(t, frequency=carrier_freq, amplitude=1.0, phase=0.0)

In [ ]:
# Visualization:
fig, ax = plt.subplots(1,1, figsize=(8,4))

ax.plot(t, c_t)

ax.grid(alpha=0.5)
ax.set_ylabel("$c(t)$")
ax.set_xlabel("$t$ in s")
ax.set_ylim(-1.1, 1.1)
ax.set_xlim(t[0], t[-1])
ax.tick_params(which="major", axis="y", direction="in")
ax.tick_params(which="both", axis="x", direction="in")

plt.show()

In the following, we will be using the triangluar carrier signal.

---
## Single-Phase PWM:
Based on the computed carrier signal, we will now implement a simple single-phase PWM.

In [ ]:
u_dc = 40  # V, exemplary dc-link voltage
u_ref_freq = 1  # Hz, exemplary targeted signal frequency
m = 1.0  # exemplary modulation index

s_ref_t = m * jnp.sin(u_ref_freq * 2 * jnp.pi * t)
u_ref_t = u_dc / 2 * s_ref_t

# compute resulting switching states:
s_t = jnp.where(s_ref_t > c_t, 1, -1)

Compute the normalized integral difference according to

$ e(t) = \frac{1}{T_\mathrm{p}} \int_{t_0}^t (s^*(\tau) - s(\tau)) \mathrm{d}\tau$,

which is approximated as

$ e(t) = \frac{1}{T_\mathrm{p}} \sum_{t_0}^t (s^*(\tau) - s(\tau)) \Delta t$,

where $\Delta t$ is the time difference between two sampling instances.

In [ ]:
delta_t = t[1] - t[0]
e_t = 1/T_p * jnp.cumsum(s_ref_t - s_t) * delta_t

In [ ]:
# Visualization:
fig, axs = plt.subplots(3,1, figsize=(12,6), sharex=True)

ax = axs[0]
ax.plot(t, s_ref_t, color="tab:blue", label="$s^*(t)$")
ax.plot(t, c_t, color="grey", alpha=0.7, label="$c(t)$")
ax.set_ylim(-1.1, 1.1)

ax.set_ylabel("$s^*(t)$,$c(t)$")
ax.legend(
    prop={"size": 16},
    framealpha=0.5,
    loc="upper center",
    fancybox=True,
    shadow=False,
    ncols=5,
)

ax = axs[1]
ax.step(t, s_t, color="tab:blue")
ax.set_ylabel("$s(t)$")
ax.set_ylim(-1.1, 1.1)


ax = axs[2]
ax.plot(t[:e_t.shape[0]], e_t, color="tab:orange")
ax.set_ylabel("$e(t)$")

for ax in axs:
    ax.grid(alpha=0.5)
    
    ax.set_xlim(t[0], t[-1])
    ax.tick_params(which="major", axis="y", direction="in")
    ax.tick_params(which="both", axis="x", direction="in")

axs[-1].set_xlabel("$t$ in s")

plt.show()

---
## Three-Phase Sinusoidal PWM:

In [ ]:
def compute_three_phase_signals(m, u_ref_freq, t, c_t, u_dc):
    omega = u_ref_freq * 2 * jnp.pi
    s_ref_t = jnp.array(
        [
            m * jnp.sin(omega * t),
            m * jnp.sin(omega * t - jnp.pi * 2 / 3),
            m * jnp.sin(omega * t + jnp.pi * 2 / 3),
        ]
    ).T
    u_ref_t = u_dc / 2  * s_ref_t

    s_t = jnp.where(s_ref_t > c_t[..., None], 1, -1)

    return u_ref_t, s_ref_t, s_t

In [ ]:
u_ref_t, s_ref_t, s_t = compute_three_phase_signals(m=0.7, u_ref_freq=2.5, t=t, c_t=c_t, u_dc=u_dc)

In [ ]:
fig_left, axs = plt.subplots(
    5, 1, figsize=(10, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1, 1, 1, 1.5]}, constrained_layout=True
)
axs[0].plot(t, s_ref_t[..., 0], color="tab:blue", label="$s^*_\\mathrm{a}$")
axs[0].plot(t, s_ref_t[..., 1], color="tab:red", label="$s^*_\\mathrm{b}$")
axs[0].plot(t, s_ref_t[..., 2], color="black", label="$s^*_\\mathrm{c}$")
axs[0].plot(t, c_t, color="grey", alpha=0.7, label="$c$")
axs[1].step(t, s_t[..., 0], color="tab:blue")
axs[2].step(t, s_t[..., 1], color="tab:red")
axs[3].step(t, s_t[..., 2], color="black")
axs[4].step(t, (s_t[..., 0] - s_t[..., 1]) / 2, color="tab:orange")


for ax in axs:
    ax.grid(alpha=0.5)
    ax.tick_params(which="major", axis="y", direction="in")
    ax.tick_params(which="both", axis="x", direction="in")
    ax.set_xlim(t[0], t[-1])

axs[0].legend(
    prop={"size": 16},
    framealpha=0.5,
    loc="upper center",
    fancybox=True,
    shadow=False,
    ncols=5,
)
axs[0].set_ylim(-1.1, 1.1)

axs[0].set_ylabel("$s^*_i(t)$,$c(t)$")
axs[1].set_ylabel("$s_\\mathrm{a}(t)$")
axs[2].set_ylabel("$s_\\mathrm{b}(t)$")
axs[3].set_ylabel("$s_\\mathrm{c}(t)$")
axs[4].set_ylabel("$\\frac{u_\\mathrm{a-b}(t)}{u_\\mathrm{dc}}$")
axs[-1].set_xlabel("$t$ in $s$")

plt.show()

### Interactive plot:

In [ ]:
def simplified_clarke_transformation(x_abc):
    T_abc2albet = jnp.array(
        [
            [2 / 3, -1 / 3, -1 / 3],
            [0, 1 / jnp.sqrt(3), -1 / jnp.sqrt(3)],
        ]
    )
    return T_abc2albet @ x_abc

def compute_pwm_space_vectors():

    all_possible_states = jnp.array(
        [
            [-1, -1, -1],
            [-1, -1, 1],
            [-1, 1, -1],
            [-1, 1, 1],
            [1, -1, -1],
            [1, -1, 1],
            [1, 1, -1],
            [1, 1, 1],
        ]
    )
    return jax.vmap(simplified_clarke_transformation)(all_possible_states)

In [ ]:
%matplotlib widget

In [ ]:
from helper_functions import InteractivePWMVisualizer

In [ ]:
u_dc = 40  # V

carrier_freq = 50  # Hz
T = 1 / carrier_freq
t = jnp.linspace(0, 20*T, 1000)

c_t = triangular_signal(t, frequency=50.0, amplitude=1.0)

If the full left half of the plot is not showing up, rerun the cell.

If some part of the plot is not showing, slightly resizing the window usually fixes the issue.

In [ ]:
plt.close('all')
plt.ioff()

visualizer = InteractivePWMVisualizer(
    t,
    c_t,
    u_dc,
    compute_three_phase_signals=jax.jit(compute_three_phase_signals),
    simplified_clarke_transformation=jax.jit(simplified_clarke_transformation),
    compute_pwm_space_vectors=compute_pwm_space_vectors,
)

display(visualizer.ui, visualizer.out)

The ["Virtual Lab"](https://homepages.thm.de/~hg13555/Datenbank/lei/index.php/de/virtuelles-labor-neu.html) from Prof. Dr.-Ing. Uwe Probst (TH Mittelhessen) provides a variety of interactive visualizations (including a similar visualization of the three-phase PWM the presented Python version is loosely based on).